# 4x3 Grid World as an MDP

In this notebook, we implement states, actions, the reward function, and the
transition model from scratch. 

We also show how to represent a policy and how to
estimate the expected utility using simulation.

**Important note:** This code is written to help understand the concepts and not for performance! I only scaled for tiny problems.

## Define the $4 \times 3$ Grid World

The used example is the simple $4 \times 3$ grid world described in AIMA Chapter 17.

![4x3 grid world description](https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/figures/AIMA_Figure_17_1.png)

AIMA Figure 17.1: (a) A simple, stochastic 
 environment that presents the agent with a sequential decision problem. (b) Illustration of the transition model of the environment: the “intended” outcome occurs with probability 0.8, but with probability 0.2 the agent moves at right angles to the intended direction. A collision with a wall results in no movement. Transitions into the two terminal states have reward +1 and -1, respectively, and all other transitions have a reward of -0.04.


 The MDP will be defined as the following global variables/functions:

* `S`: set of states.
* `A`: set of actions.
* `p(sp, s, a)`: a function returning the transition probability $P(s' | s, a)$.
* `r(s, a, sp)`: reward function for the transition from $s$ to $s`$ with 
  action `a`.

Policies are represented as:

* Deterministic policies are a vector with the action for each state.
* Stochastic policies are a matrix with probabilities where each row is a 
  state and the columns are the actions 

## States

We define the atomic state space $S$ by labeling the states $0, 2, ...$.
We convert coordinates `(rows, columns)` to the state label.

In [1]:
import numpy as np

In [2]:
# I use capitalized variables as global constants
COLS = 4
ROWS = 3

S = np.array(range(ROWS * COLS))
S

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [3]:
def list_to_layout(x):
    return(np.array(x).reshape(ROWS, COLS, order = "F")[::-1])

LAYOUT = list_to_layout(S)
LAYOUT

array([[ 2,  5,  8, 11],
       [ 1,  4,  7, 10],
       [ 0,  3,  6,  9]])

Convert between the state description as coordinates and state labels.

In [4]:
def coord_to_s(coord):
  assert coord[0] in range(LAYOUT.shape[0])
  assert coord[1] in range(LAYOUT.shape[1])

  s = list_to_layout(LAYOUT)[coord]
  return s

def s_to_coord(s):
    assert s in S

    coord = np.where(LAYOUT[::-1] == s)
    return coord

# check with top-right corner
print("Coord for state 11:", s_to_coord(11))
print("State at coord (2,3):", coord_to_s((2,3)))


Coord for state 11: (array([2]), array([3]))
State at coord (2,3): 11


Note: Coordinates are formatted `(column,row)`

Define the start and the terminal states.

In [5]:
START = (0, )

GOAL, TRAP = 11, 10
TERMINAL = (GOAL, TRAP)

In [6]:
def is_terminal(s):
    return s in TERMINAL

is_terminal(GOAL)

True

In [7]:
UNREACHABLE = (4, )

def is_reachable(s):
    return not s in UNREACHABLE and s in S 

print(is_reachable(0)) 
print(is_reachable(4)) 
display(is_reachable(100)) 

True
False


False

## Actions

The complete set of actions is
$A = \{\mathrm{'Up', 'Right', 'Down', 'Left'}\}$.



In [8]:
A = ['up', 'right', 'down', 'left']

In [9]:
A_lookup = {
    "up": 0,
    "right": 1,
    "down": 2,
    "left": 3
}

## Transition Model

$P(s' | s, a)$ is the probability of going from state $s$ to $s'$ by
when taking action $a$. We will create a function $p(s, a, s')$.


In [10]:
def move(s, a):
    r, c = s_to_coord(s)
    a_ind = A.index(a)
    
    r, c = {
        0: (r+1, c),
        1: (r,   c+1),
        2: (r-1, c), 
        3: (r,   c-1),
    }[a_ind]
    
    if r<0: r=0
    if r>=LAYOUT.shape[0]: r=LAYOUT.shape[0]-1
    if c<0: c=0
    if c>=LAYOUT.shape[1]: c=LAYOUT.shape[1]-1

    sp = coord_to_s((r, c))
    
    # this is for the wall block at (1,1)
    if not is_reachable(sp):
        return s
    
    return sp

# choose actions 90 degrees to the right or to the left of a 
def turn(a, dir):
    a_ind = A.index(a)
    a_turn = a_ind+dir
    
    if a_turn > len(A)-1:
        a_turn = 0
    if a_turn < 0:
        a_turn = len(A)-1
    
    return A[a_turn]

In [11]:

def p(s, a, sp):
    assert a in A
    assert s in S
    assert sp in S 
 
    prob = 0.0
    
    if sp == move(s, a):
        prob += 0.8
    
    if sp == move(s, turn(a, +1)):
        prob += 0.1
    
    if sp == move(s, turn(a, -1)):
        prob += 0.1
    
    return prob

Probability over states after moving up from the bottom-left corner.

In [12]:
list_to_layout([p(0, "up", s) for s in S])

array([[0. , 0. , 0. , 0. ],
       [0.8, 0. , 0. , 0. ],
       [0.1, 0.1, 0. , 0. ]])

Represent the transition function as a transition matrix. Here we show the example for the action "up".

In [27]:
np.array([p(s, "up", s_p) for s in S for s_p in S]).reshape((len(S), len(S)))
        

array([[0.1, 0.8, 0. , 0.1, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0.2, 0.8, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0.9, 0. , 0. , 0.1, 0. , 0. , 0. , 0. , 0. , 0. ],
       [0.1, 0. , 0. , 0.8, 0. , 0. , 0.1, 0. , 0. , 0. , 0. , 0. ],
       [0. , 0.1, 0. , 0. , 0. , 0.8, 0. , 0.1, 0. , 0. , 0. , 0. ],
       [0. , 0. , 0.1, 0. , 0. , 0.8, 0. , 0. , 0.1, 0. , 0. , 0. ],
       [0. , 0. , 0. , 0.1, 0. , 0. , 0. , 0.8, 0. , 0.1, 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0.8, 0. , 0.1, 0. ],
       [0. , 0. , 0. , 0. , 0. , 0.1, 0. , 0. , 0.8, 0. , 0. , 0.1],
       [0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0. , 0. , 0.1, 0.8, 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0. , 0. , 0.1, 0.8],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0. , 0. , 0.9]])

## Reward Model

$r(s, a, s')$ define the reward for the transition from $s$ to $s'$ with
action $a$.

For the textbook example we have:

-   Any move costs utility (a reward of -0.04).
-   Going to state 12 has a reward of +1
-   Going to state 11 has a reward of -1.

Note that once you are in an absorbing state (11 or 12), then the
problem is over and there is no more reward!


In [13]:
def r(s, a, sp):
    ## no more reward when we in 11 or 12.
    if s == GOAL or s == TRAP: 
        return 0.0
  
    ## transition to the absorbing states.
    if sp == GOAL: 
        return +1.0
  
    if sp == TRAP:
        return -1.0
  
    ## cost for each move
    return -0.04

In [14]:
r(0,"up", 1), r(8,"right", 11), r(7,"right", 10)

(-0.04, 1.0, -1.0)

Example reward matrix for the action "up".

In [28]:
np.array([r(s, "up", s_p) for s in S for s_p in S]).reshape((len(S), len(S)))

array([[-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  ,  1.  ],
       [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04,
        -0.04, -1.  

Note: many s, a, s' combinations are not possible (have a 0 in the p matrix)

## Policy
The solution to an MDP is a policy $\pi$ which defines which action
to take in each state. 

### Deterministic Policies

It can be shown that all MPDs have an optimal deterministic policy with one 
action per state.
We represent deterministic policies as a vector of
actions.
I make up a policy that always goes up and then to the right once the
agent hits the top.

In [15]:
pi_manual = ['up'] * len(S)
pi_manual[2] = 'right'
pi_manual[5] = 'right'
pi_manual[8] = 'right'
pi_manual

#show_layout(pi_manual)

['up',
 'up',
 'right',
 'up',
 'up',
 'right',
 'up',
 'up',
 'right',
 'up',
 'up',
 'up']

In [16]:
list_to_layout(pi_manual)

array([['right', 'right', 'right', 'up'],
       ['up', 'up', 'up', 'up'],
       ['up', 'up', 'up', 'up']], dtype='<U5')

We can also create a random policy by randomly choosing from the
available actions for each state.

In [17]:
def create_random_deterministic_policy():
    return np.random.choice(A, size=len(S))

In [18]:
pi_random = create_random_deterministic_policy()
list_to_layout(pi_random)

array([['right', 'left', 'down', 'up'],
       ['up', 'up', 'left', 'left'],
       ['down', 'down', 'right', 'up']], dtype='<U5')

### Stochastic Policies

Stochastic policies use probabilities of actions in each state. 
They are defined as the function $\pi(a|s)$.
We use as simple table with probabilities where each row is a state and the columns 
are the actions. The rows have to add up to 1.


Here we create a random $\epsilon$-soft policy. Each available has
at least a probability of $\epsilon$ divided by the number of actions.

In [19]:
def make_policy_epsilon_soft (pol, epsilon=0.1):
    pol = [A_lookup[a] for a in pol]
    
    p = np.zeros((len(S), len(A)))
    p.fill(epsilon / len(A))

    for s in S:
        p[s, pol[s]] = p[s, pol[s]] + (1 - epsilon)
     
    return p

In [20]:
make_policy_epsilon_soft(pi_manual, epsilon = 0.1)

array([[0.925, 0.025, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.025, 0.925, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.025, 0.925, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.025, 0.925, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025],
       [0.925, 0.025, 0.025, 0.025]])

## Value Function

### Definition

The value function given a policy $\pi$ is defined as the expected utility:

$U^\pi = \mathbb{E}\left[\sum_{t=0}^\infty \gamma^t R(s_t, \pi(s_t), s_{t+1})\right]$

We need to define the discount factor.

In [21]:
GAMMA = 1

### Estimation of the Expected Utility

We can simulate (sample) a trajectory and calculate the discounted return of a single trajectory.

$G = \sum_{t=0}^\infty \gamma^t R(s_t, \pi(s_t), s_{t+1})$

All rewards after the terminal state are defined as zero, so we can stop when a terminal state is reached.

In [22]:
def sample(pol, start, max_steps = 1000, verbose=False):
    s = start
    ret = 0.0
    df = 1
    
    if verbose:
        print("step: s, a, sp, reward")

    step = 0  # use max_step for policies that do not lead to a terminal state 
    while not is_terminal(s) or step > max_steps:
        a = pol[s]

        s_probs = [p(s, a, sp) for sp in S]
        sp = np.random.choice(S,p=s_probs)

        rew = r(s,a,sp)
        ret += df * rew
        df *= GAMMA

        if verbose:
            print(f"{step}: {s}, {a}, {sp}, {rew}")

        s = sp
        step += 1

    return ret

In [23]:
sample(pi_manual, 0, verbose = True)

step: s, a, sp, reward
0: 0, up, 0, -0.04
1: 0, up, 1, -0.04
2: 1, up, 2, -0.04
3: 2, right, 5, -0.04
4: 5, right, 8, -0.04
5: 8, right, 8, -0.04
6: 8, right, 11, 1.0


0.76

The expected utility of a state given a policy can be estimated by averaging the return of many sampled trajectories.
This is called a Monte Carlo simulation.

In [24]:
returns = [sample(pi_manual, 0) for _ in range(100)]
returns

[0.6799999999999999,
 0.76,
 0.76,
 0.84,
 0.76,
 0.4,
 0.8,
 0.76,
 0.8,
 0.8,
 0.52,
 0.76,
 0.6799999999999999,
 0.76,
 0.8,
 0.8,
 -1.32,
 0.72,
 0.56,
 0.76,
 0.84,
 0.8,
 0.84,
 0.8,
 0.84,
 0.84,
 0.84,
 0.76,
 0.72,
 0.8,
 0.72,
 0.84,
 0.76,
 0.84,
 0.84,
 0.84,
 0.8,
 0.76,
 0.8,
 0.8,
 0.84,
 0.84,
 0.8,
 0.8,
 0.72,
 0.84,
 -1.2,
 0.8,
 -1.4,
 0.64,
 0.8,
 0.8,
 0.76,
 0.84,
 0.8,
 0.8,
 0.84,
 0.6799999999999999,
 -1.24,
 0.72,
 0.8,
 0.64,
 0.6000000000000001,
 0.8,
 0.64,
 0.8,
 0.72,
 0.84,
 0.84,
 0.64,
 0.8,
 0.76,
 0.8,
 0.76,
 0.6799999999999999,
 0.76,
 0.56,
 0.84,
 0.72,
 0.84,
 0.84,
 0.84,
 0.8,
 0.84,
 0.6799999999999999,
 0.8,
 0.84,
 0.8,
 0.84,
 0.84,
 0.84,
 0.76,
 0.84,
 0.84,
 0.72,
 0.76,
 0.84,
 0.84,
 0.76,
 0.76]

In [25]:
np.mean(returns)

np.float64(0.6888000000000001)

The average is a good estimate for the expected return.